# Healthcare claims pipeline

Open this notebook in **hosted Google Colab** in your browser. Run each section from top to bottom.

You will choose three files, one at a time:
- **Step 2:** `C:\Users\bidus\pipeline_data\landing.zip`
- **Step 3:** `src/claims_pipeline/bronze.py`
- **Step 4:** `src/claims_pipeline/silver.py` from your local `claims-pipeline` project

Steps 1 and 5 have no uploads. Step 6 downloads the cleaned CSVs for Snowflake.

## 1. Install PySpark and start Spark

In [ ]:
%pip install -q pyspark pytest

from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[2]").appName("ClaimsPipeline").getOrCreate()
print("Spark version:", spark.version)

## 2. Upload the batch ZIP

Choose **landing.zip**. Expect **Found 107 CSVs**.

In [ ]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

from google.colab import files

print(r"Choose C:\Users\bidus\pipeline_data\landing.zip")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Choose one file: landing.zip.")

filename, contents = next(iter(uploaded.items()))
if not filename.lower().endswith(".zip"):
    raise ValueError("Choose landing.zip.")

landing = Path("/content/claims_input")
with ZipFile(BytesIO(contents)) as archive:
    archive.extractall(landing)

csv_files = list(landing.rglob("*.csv"))
print(f"Found {len(csv_files)} CSVs")

## 3. Upload bronze.py and build bronze Parquet

Run the first cell and choose **src/claims_pipeline/bronze.py** from your project folder. In a fresh runtime, Run 1 writes **107** files and Run 2 skips all **107**. In a runtime with existing bronze outputs, both runs can skip all 107.

Run the second cell to read bronze. Expect **9,769 claim rows** and **89,474 transaction rows**.

In [ ]:
from importlib import reload

from pyspark.sql import functions as F

package = Path("src/claims_pipeline")
package.mkdir(parents=True, exist_ok=True)
(package / "__init__.py").touch(exist_ok=True)

print("Choose src/claims_pipeline/bronze.py from your project folder.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Choose one file: bronze.py.")

filename, contents = next(iter(uploaded.items()))
if not filename.startswith("bronze") or not filename.endswith(".py"):
    raise ValueError("Choose src/claims_pipeline/bronze.py.")
(package / "bronze.py").write_bytes(contents)

from src.claims_pipeline import bronze as bronze_job

reload(bronze_job)
bronze = Path("/content/bronze")

for attempt in (1, 2):
    written, skipped = bronze_job.run_bronze(spark, landing, bronze)
    print(f"Run {attempt}: Files written: {written} | Files skipped: {skipped}")


In [7]:
def read_bronze(pattern):
    paths = [str(path) for path in sorted(bronze.glob(pattern))]
    return spark.read.option("mergeSchema", "true").parquet(*paths)

bronze_claims = read_bronze("claims_*")
bronze_transactions = read_bronze("transactions_*")

print("Bronze claim rows:", bronze_claims.count())
print("Bronze transaction rows:", bronze_transactions.count())

Bronze claim rows: 9769
Bronze transaction rows: 89474


## 4. Upload silver.py and clean the bronze data

Choose **src/claims_pipeline/silver.py** from your project folder. The notebook imports this file and runs the cleaning code.

Expect **541 duplicate claim rows removed**, **9,227 valid claims**, **89,473 valid transactions**, and **one quarantined row in each table**.

In [ ]:
from importlib import reload

package = Path("src/claims_pipeline")
package.mkdir(parents=True, exist_ok=True)
(package / "__init__.py").touch(exist_ok=True)

print("Choose src/claims_pipeline/silver.py from your project folder.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Choose one file: silver.py.")

filename, contents = next(iter(uploaded.items()))
if not filename.startswith("silver") or not filename.endswith(".py"):
    raise ValueError("Choose src/claims_pipeline/silver.py.")
(package / "silver.py").write_bytes(contents)

from src.claims_pipeline import silver as silver_job

reload(silver_job)
silver_path = "/content/silver"

checked_claims = silver_job.clean_claims(bronze_claims)
checked_transactions = silver_job.clean_transactions(bronze_transactions)

duplicates = bronze_claims.count() - checked_claims.count()
print(f"Claim rows removed as duplicates: {duplicates:,}")

for table, checked in [
    ("claims", checked_claims),
    ("transactions", checked_transactions),
]:
    valid, quarantined = silver_job.write_silver(table, checked, silver_path)
    print(f"{table}: {valid:,} valid | {quarantined:,} quarantined")

## 5. Check the silver results

Expect **14 transactions without a valid claim** and **3 passing pytest checks**.

In [10]:
silver_claims = spark.read.parquet("/content/silver/claims")
silver_transactions = spark.read.parquet("/content/silver/transactions")

claim_ids = silver_claims.select(F.col("Id").alias("CLAIMID"))
unmatched = silver_transactions.join(claim_ids, on="CLAIMID", how="left_anti")

print("Transactions without a valid claim:", unmatched.count())

Transactions without a valid claim: 14


In [12]:
%%writefile /content/test_silver.py
import pytest
from pyspark.sql import SparkSession


@pytest.fixture(scope="module")
def spark():
    return SparkSession.builder.getOrCreate()


def test_silver_claims(spark):
    claims = spark.read.parquet("/content/silver/claims")
    assert claims.count() == 9_227
    assert claims.select("Id").distinct().count() == 9_227
    assert claims.filter("Id IS NULL").count() == 0


def test_silver_transactions(spark):
    transactions = spark.read.parquet("/content/silver/transactions")
    assert transactions.count() == 89_473
    assert transactions.filter(
        "TYPE IN ('CHARGE', 'TRANSFERIN') AND amount_decimal IS NULL"
    ).count() == 0


def test_quarantine(spark):
    assert spark.read.parquet("/content/silver/quarantine/claims").count() == 1
    assert spark.read.parquet("/content/silver/quarantine/transactions").count() == 1

Writing /content/test_silver.py


In [13]:
import pytest

pytest.main(["-q", "/content/test_silver.py"])

...                                                                      [100%]
3 passed in 3.02s


<ExitCode.OK: 0>

## 6. Download the silver CSVs for Snowflake

Allow both downloads if your browser asks. Upload **silver_claims.csv** and **silver_transactions.csv** separately into Snowflake's `CLAIMS_PIPELINE.STAGING` schema as `SILVER_CLAIMS` and `SILVER_TRANSACTIONS`. Use `sql/gold_pipeline.sql` in Snowflake to build and report on the gold tables.

In [11]:
from google.colab import files

claims = spark.read.parquet("/content/silver/claims").selectExpr(
    "Id AS CLAIM_ID",
    "PATIENTID AS PATIENT_KEY",
    "PROVIDERID AS PROVIDER_ID",
    "substr(SERVICEDATE, 1, 10) AS SERVICE_DATE",
    "STATUS1 AS CLAIM_STATUS",
    "batch_id AS BATCH_ID",
)

transactions = spark.read.parquet("/content/silver/transactions").selectExpr(
    "ID AS TRANSACTION_ID",
    "CLAIMID AS CLAIM_ID",
    "TYPE AS TRANSACTION_TYPE",
    "amount_decimal AS AMOUNT",
    "try_cast(PAYMENTS AS DECIMAL(18, 2)) AS PAYMENT_AMOUNT",
    "batch_id AS BATCH_ID",
)

for name, table in [("silver_claims", claims), ("silver_transactions", transactions)]:
    path = f"/content/{name}.csv"
    table.toPandas().to_csv(path, index=False)
    files.download(path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>